# 02 - Pipeline de données

> Notebook de formalisation du pipeline reproductible du projet eau.

Ce notebook présente la logique du pipeline sous GitHub, en séparant clairement les couches staging, intermediate et marts, avec une conclusion pour chaque bloc.

## Objectif et sommaire

> Objectif : expliciter un pipeline stable, versionnable et réexécutable à partir des fichiers bruts jusqu'aux tables prêtes pour les usages métier.

Lecture recommandée :

1. rappeler les entrées et sorties du pipeline ;
2. décrire la couche staging et ses contrôles ;
3. décrire la couche intermediate et ses transformations ;
4. décrire la couche marts et les jeux de données diffusables ;
5. conclure sur la reproductibilité et la contrainte WASH 2016.

In [10]:
from src.schema_fr import resolve_project_paths
import pandas as pd

paths = resolve_project_paths()
PROJECT_ROOT = paths['PROJECT_ROOT']
RAW_DATA_PATH = paths['RAW_DATA_PATH']
PROCESSED_DATA_PATH = paths['PROCESSED_DATA_PATH']
CSV_EN_PATH = paths['CSV_EN_PATH']
CSV_FR_PATH = paths['CSV_FR_PATH']
PBI_STAR_PATH = paths['PBI_STAR_PATH']

raw_files = sorted(path.name for path in RAW_DATA_PATH.glob('*.csv'))
processed_inventory = pd.DataFrame(
    [
        {'zone': 'csv_enrichis_en', 'exists': CSV_EN_PATH.exists(), 'files': sorted(path.name for path in CSV_EN_PATH.glob('*.csv')) if CSV_EN_PATH.exists() else []},
        {'zone': 'csv_enrichis_fr', 'exists': CSV_FR_PATH.exists(), 'files': sorted(path.name for path in CSV_FR_PATH.glob('*.csv')) if CSV_FR_PATH.exists() else []},
        {'zone': 'pbi_star', 'exists': PBI_STAR_PATH.exists(), 'files': sorted(path.name for path in PBI_STAR_PATH.glob('*.csv')) if PBI_STAR_PATH.exists() else []},
    ]
)

print('Fichiers raw :', raw_files)
display(processed_inventory)

Fichiers raw : ['BasicAndSafelyManagedDrinkingWaterServices.csv', 'MortalityRateAttributedToWater.csv', 'PoliticalStability.csv', 'Population.csv', 'RegionCountry.csv']


,zone,exists,files
0,csv_enrichis_en,True,"[country_region_reference.csv, dashboard_water..."
1,csv_enrichis_fr,True,"[country_region_reference_fr.csv, dashboard_wa..."
2,pbi_star,True,"[dim_action_star_fr.csv, dim_annee_star_fr.csv..."


<a id="RNCP37837BC02-STAGING"></a>

## Partie 1 - Couche staging

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Bloc compétence :</strong> <span style="background-color: #E8F4F8; padding: 2px 6px; border-radius: 3px;">RNCP37837BC02</span> — Extraire et agréger · Traiter les données manquantes
</div>

### Définition

La couche staging correspond à l'atterrissage contrôlé des données brutes. On ne cherche pas encore à faire de l'analyse métier ; on vérifie surtout la structure, la présence des colonnes attendues, les types et les anomalies visibles.

In [11]:
staging_overview = pd.DataFrame(
    [
        {'table': 'Population.csv', 'role': 'contexte démographique', 'grain': 'Country-Year-Granularity'},
        {'table': 'RegionCountry.csv', 'role': 'référentiel pays-régions', 'grain': 'Country'},
        {'table': 'PoliticalStability.csv', 'role': 'stabilité politique', 'grain': 'Country-Year-Granularity'},
        {'table': 'MortalityRateAttributedToWater.csv', 'role': 'vulnérabilité sanitaire WASH', 'grain': 'Country-Year-Granularity'},
        {'table': 'BasicAndSafelyManagedDrinkingWaterServices.csv', 'role': 'accès et qualité des services deau', 'grain': 'Country-Year-Granularity'},
    ]
)
staging_overview

,table,role,grain
0,Population.csv,contexte démographique,Country-Year-Granularity
1,RegionCountry.csv,référentiel pays-régions,Country
2,PoliticalStability.csv,stabilité politique,Country-Year-Granularity
3,MortalityRateAttributedToWater.csv,vulnérabilité sanitaire WASH,Country-Year-Granularity
4,BasicAndSafelyManagedDrinkingWaterServices.csv,accès et qualité des services deau,Country-Year-Granularity


### Conclusion du bloc staging

La couche staging protège le pipeline contre les erreurs de format. C'est ici qu'on détecte les écarts de noms de pays, l'unité de population en milliers, et la limite majeure de la mortalité WASH disponible uniquement pour 2016.

<a id="RNCP37837BC02-INTERMEDIATE"></a>

## Partie 2 - Couche intermediate

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Compétence :</strong> <span style="background-color: #E8F4F8; padding: 2px 6px; border-radius: 3px;">Vérifier la cohérence</span> — Standardisation pays, rattachement régions, conversion d'unités
</div>

### Définition

La couche intermediate transforme les sources brutes en tables harmonisées. C'est là que la logique métier et la logique de qualité sont centralisées : standardisation des pays, rattachement aux macro-régions, conversion des unités et normalisation des indicateurs.

In [12]:
intermediate_candidates = {
    'country_region_reference_en': CSV_EN_PATH / 'country_region_reference.csv',
    'country_region_reference_fr': CSV_FR_PATH / 'country_region_reference_fr.csv',
    'water_indicators_long_en': CSV_EN_PATH / 'water_indicators_long.csv',
    'water_indicators_long_fr': CSV_FR_PATH / 'water_indicators_long_fr.csv',
}

intermediate_summary = []
for dataset_name, dataset_path in intermediate_candidates.items():
    exists = dataset_path.exists()
    rows = None
    columns = None
    if exists:
        temp_df = pd.read_csv(dataset_path, sep=';')
        rows = len(temp_df)
        columns = list(temp_df.columns)
    intermediate_summary.append({
        'dataset': dataset_name,
        'exists': exists,
        'rows': rows,
        'columns': columns,
    })

pd.DataFrame(intermediate_summary)

,dataset,exists,rows,columns
0,country_region_reference_en,True,240,"[country_clean, region]"
1,country_region_reference_fr,True,240,"[pays, region]"
2,water_indicators_long_en,True,46490,"[country, region, year, granularity, value, in..."
3,water_indicators_long_fr,True,46490,"[pays, region, annee, granularite, valeur, ind..."


### Conclusion du bloc intermediate

La couche intermediate constitue le coeur du pipeline. Elle fabrique des tables réutilisables dans plusieurs usages, sans dépendre directement d'un outil de visualisation. C'est aussi ici que la contrainte 2016 de la mortalité doit rester explicite pour éviter des analyses temporelles trompeuses.

## Partie 3 - Couche marts

### Définition

Les marts sont des tables orientées usage. Elles simplifient la consommation des données pour les analyses métier, Tableau, Power BI ou une soutenance. Elles portent déjà la granularité d'analyse choisie, par exemple pays-année ou snapshot de priorisation.

Dans ce projet, la publication des marts suit une règle simple : une seule logique métier amont en couche intermediate, puis plusieurs vues publiées en aval. Cela permet de diffuser deux versions sœurs, l'une en anglais et l'autre en français, sans dupliquer les règles de calcul ni altérer la conservation de l'information métier, y compris sur le cas de l'agrégat Chine.

In [13]:
mart_candidates = {
    'dashboard_water_country_year_en': CSV_EN_PATH / 'dashboard_water_country_year.csv',
    'priority_snapshot_2016_en': CSV_EN_PATH / 'priority_snapshot_2016.csv',
    'priority_snapshot_focus_2016_en': CSV_EN_PATH / 'priority_snapshot_focus_2016.csv',
    'dashboard_water_country_year_fr': CSV_FR_PATH / 'dashboard_water_country_year_fr.csv',
    'priority_snapshot_2016_fr': CSV_FR_PATH / 'priority_snapshot_2016_fr.csv',
    'priority_snapshot_focus_2016_fr': CSV_FR_PATH / 'priority_snapshot_focus_2016_fr.csv',
    'fact_dashboard_star_fr': PBI_STAR_PATH / 'fact_dashboard_star_fr.csv',
}

mart_summary = []
for dataset_name, dataset_path in mart_candidates.items():
    exists = dataset_path.exists()
    rows = None
    columns = None
    if exists:
        temp_df = pd.read_csv(dataset_path, sep=';')
        rows = len(temp_df)
        columns = list(temp_df.columns)
    mart_summary.append({
        'dataset': dataset_name,
        'exists': exists,
        'rows': rows,
        'columns': columns,
    })

pd.DataFrame(mart_summary)

,dataset,exists,rows,columns
0,dashboard_water_country_year_en,True,4466,"[country, region, year, population_total_peopl..."
1,priority_snapshot_2016_en,True,235,"[country, region, year, population_total_peopl..."
2,priority_snapshot_focus_2016_en,True,173,"[country, region, year, population_total_peopl..."
3,dashboard_water_country_year_fr,True,4466,"[pays, region, annee, population_totale, popul..."
4,priority_snapshot_2016_fr,True,235,"[pays, region, annee, population_totale, popul..."
5,priority_snapshot_focus_2016_fr,True,173,"[pays, region, annee, population_totale, popul..."
6,fact_dashboard_star_fr,True,4466,"[cle_pays, cle_annee, cle_action, pays, region..."


In [19]:
mart_pairs = [
    ('dashboard_water_country_year', CSV_EN_PATH / 'dashboard_water_country_year.csv', CSV_FR_PATH / 'dashboard_water_country_year_fr.csv'),
    ('priority_snapshot_2016', CSV_EN_PATH / 'priority_snapshot_2016.csv', CSV_FR_PATH / 'priority_snapshot_2016_fr.csv'),
    ('priority_snapshot_focus_2016', CSV_EN_PATH / 'priority_snapshot_focus_2016.csv', CSV_FR_PATH / 'priority_snapshot_focus_2016_fr.csv'),
]

publication_audit = []
for mart_name, en_path, fr_path in mart_pairs:
    en_exists = en_path.exists()
    fr_exists = fr_path.exists()
    en_rows = fr_rows = None
    en_columns = fr_columns = None
    same_row_count = None
    same_column_count = None

    if en_exists:
        en_df = pd.read_csv(en_path, sep=';')
        en_rows = len(en_df)
        en_columns = len(en_df.columns)
    if fr_exists:
        fr_df = pd.read_csv(fr_path, sep=';')
        fr_rows = len(fr_df)
        fr_columns = len(fr_df.columns)

    if en_exists and fr_exists:
        same_row_count = en_rows == fr_rows
        same_column_count = en_columns == fr_columns

    publication_audit.append({
        'mart': mart_name,
        'en_exists': en_exists,
        'fr_exists': fr_exists,
        'en_rows': en_rows,
        'fr_rows': fr_rows,
        'same_row_count': same_row_count,
        'en_column_count': en_columns,
        'fr_column_count': fr_columns,
        'same_column_count': same_column_count,
    })

pbi_fact_path = PBI_STAR_PATH / 'fact_dashboard_star_fr.csv'
pbi_fact_exists = pbi_fact_path.exists()
pbi_fact_rows = None
pbi_fact_columns = None
if pbi_fact_exists:
    pbi_fact_df = pd.read_csv(pbi_fact_path, sep=';')
    pbi_fact_rows = len(pbi_fact_df)
    pbi_fact_columns = len(pbi_fact_df.columns)

display(pd.DataFrame(publication_audit))
display(
    pd.DataFrame(
        [
            {
                'contract': 'pbi_star',
                'exists': pbi_fact_exists,
                'rows': pbi_fact_rows,
                'column_count': pbi_fact_columns,
                'role': 'contrat technique distinct pour Power BI',
            }
        ]
    )
)

,mart,en_exists,fr_exists,en_rows,fr_rows,same_row_count,en_column_count,fr_column_count,same_column_count
0,dashboard_water_country_year,True,True,4466,4466,True,22,22,True
1,priority_snapshot_2016,True,True,235,235,True,22,22,True
2,priority_snapshot_focus_2016,True,True,173,173,True,22,22,True


,contract,exists,rows,column_count,role
0,pbi_star,True,4466,25,contrat technique distinct pour Power BI


### Audit des contrats de publication

Le contrôle ci-dessous vérifie que les marts analytiques EN et FR sont bien publiés comme deux vues sœurs d'un même socle intermediate. L'objectif n'est pas de comparer les libellés mot à mot, mais de s'assurer que le volume et la structure générale des sorties restent alignés, tandis que `pbi_star` demeure un contrat séparé.

In [15]:
grain_audit = []

grain_pairs = [
    ('dashboard_water_country_year', CSV_EN_PATH / 'dashboard_water_country_year.csv', CSV_FR_PATH / 'dashboard_water_country_year_fr.csv'),
    ('priority_snapshot_2016', CSV_EN_PATH / 'priority_snapshot_2016.csv', CSV_FR_PATH / 'priority_snapshot_2016_fr.csv'),
    ('priority_snapshot_focus_2016', CSV_EN_PATH / 'priority_snapshot_focus_2016.csv', CSV_FR_PATH / 'priority_snapshot_focus_2016_fr.csv'),
]

for mart_name, en_path, fr_path in grain_pairs:
    en_df = pd.read_csv(en_path, sep=';')
    fr_df = pd.read_csv(fr_path, sep=';')

    en_keys = ['country', 'region', 'year']
    fr_keys = ['pays', 'region', 'annee']

    en_duplicate_keys = int(en_df.duplicated(subset=en_keys).sum())
    fr_duplicate_keys = int(fr_df.duplicated(subset=fr_keys).sum())

    en_distinct_keys = len(en_df[en_keys].drop_duplicates())
    fr_distinct_keys = len(fr_df[fr_keys].drop_duplicates())

    grain_audit.append({
        'mart': mart_name,
        'grain_en': 'country-region-year',
        'grain_fr': 'pays-region-annee',
        'distinct_keys_en': en_distinct_keys,
        'distinct_keys_fr': fr_distinct_keys,
        'same_distinct_key_count': en_distinct_keys == fr_distinct_keys,
        'duplicate_keys_en': en_duplicate_keys,
        'duplicate_keys_fr': fr_duplicate_keys,
        'same_duplicate_count': en_duplicate_keys == fr_duplicate_keys,
        'min_year_en': int(en_df['year'].min()),
        'max_year_en': int(en_df['year'].max()),
        'min_year_fr': int(fr_df['annee'].min()),
        'max_year_fr': int(fr_df['annee'].max()),
        'same_year_range': (
            int(en_df['year'].min()) == int(fr_df['annee'].min())
            and int(en_df['year'].max()) == int(fr_df['annee'].max())
        ),
    })

pd.DataFrame(grain_audit)

,mart,grain_en,grain_fr,distinct_keys_en,distinct_keys_fr,same_distinct_key_count,duplicate_keys_en,duplicate_keys_fr,same_duplicate_count,min_year_en,max_year_en,min_year_fr,max_year_fr,same_year_range
0,dashboard_water_country_year,country-region-year,pays-region-annee,4466,4466,True,0,0,True,2000,2018,2000,2018,True
1,priority_snapshot_2016,country-region-year,pays-region-annee,235,235,True,0,0,True,2016,2016,2016,2016,True
2,priority_snapshot_focus_2016,country-region-year,pays-region-annee,173,173,True,0,0,True,2016,2016,2016,2016,True


### Audit de granularité et des clés métier

Ce second contrôle vérifie que les marts analytiques EN et FR portent bien la même granularité métier. On compare ici la clé logique `pays-région-année`, le nombre de doublons sur cette clé et la plage temporelle couverte, afin de confirmer que la version française reste une publication jumelle et non une variante métier.

In [16]:
import importlib
import src.schema_fr as schema_fr
importlib.reload(schema_fr)

marts_contract = schema_fr.construire_marts_analytiques_en_fr()
semantic_audit = []

semantic_pairs = [
    ('dashboard_water_country_year', CSV_EN_PATH / 'dashboard_water_country_year.csv', CSV_FR_PATH / 'dashboard_water_country_year_fr.csv'),
    ('priority_snapshot_2016', CSV_EN_PATH / 'priority_snapshot_2016.csv', CSV_FR_PATH / 'priority_snapshot_2016_fr.csv'),
    ('priority_snapshot_focus_2016', CSV_EN_PATH / 'priority_snapshot_focus_2016.csv', CSV_FR_PATH / 'priority_snapshot_focus_2016_fr.csv'),
]

for mart_name, en_path, fr_path in semantic_pairs:
    en_columns = list(pd.read_csv(en_path, sep=';', nrows=0).columns)
    fr_columns = list(pd.read_csv(fr_path, sep=';', nrows=0).columns)
    expected_mapping = marts_contract[mart_name]
    translated_en_columns = [expected_mapping[column] for column in en_columns]

    semantic_audit.append({
        'mart': mart_name,
        'same_column_order': en_columns == list(expected_mapping.keys()),
        'same_translated_contract': translated_en_columns == fr_columns,
        'contract_column_count': len(expected_mapping),
        'en_column_count': len(en_columns),
        'fr_column_count': len(fr_columns),
        'unexpected_en_columns': sorted(set(en_columns) - set(expected_mapping.keys())),
        'unexpected_fr_columns': sorted(set(fr_columns) - set(expected_mapping.values())),
    })

pd.DataFrame(semantic_audit)

,mart,same_column_order,same_translated_contract,contract_column_count,en_column_count,fr_column_count,unexpected_en_columns,unexpected_fr_columns
0,dashboard_water_country_year,True,True,22,22,22,[],[]
1,priority_snapshot_2016,True,True,22,22,22,[],[]
2,priority_snapshot_focus_2016,True,True,22,22,22,[],[]


### Audit sémantique du contrat EN/FR

Ce troisième contrôle vérifie que les colonnes des marts EN et FR se correspondent bien selon un contrat de traduction explicite. On ne compare donc plus seulement les volumes ou la granularité, mais l'alignement attendu des noms de colonnes publiés dans chaque langue.

<a id="RNCP37837BC01"></a>

## Partie — Couche marts (Star Schema Power BI)

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Bloc compétence :</strong> <span style="background-color: #E8F5E9; padding: 2px 6px; border-radius: 3px;">RNCP37837BC01</span> — Créer une base de données · Gérer une base de données
</div>

In [ ]:
dashboard_design_contract = pd.DataFrame(
    [
        {
            'page_dashboard': 'Page 2 - Domaines metiers',
            'composant': 'Courbe temporelle Domaine 1',
            'granularite_recommandee': 'region',
            'arbitrage': 'utiliser les regions pour montrer une evolution lisible dans le temps et eviter un spaghetti plot pays',
        },
        {
            'page_dashboard': 'Page 2 - Domaines metiers',
            'composant': 'Barres empilees structure urbaine/rurale',
            'granularite_recommandee': 'country',
            'arbitrage': 'conserver les pays prioritaires car la structure urbaine ou rurale doit rester actionnable',
        },
        {
            'page_dashboard': 'Page 3 - Priorisation',
            'composant': 'Top pays et recommandations',
            'granularite_recommandee': 'country',
            'arbitrage': 'le pays reste l unite de decision ; la region sert de filtre, regroupement ou contexte secondaire',
        },
        {
            'page_dashboard': 'Page 3 - Priorisation',
            'composant': 'Vue de contexte macro',
            'granularite_recommandee': 'region',
            'arbitrage': 'les regions peuvent resumer les tendances moyennes, sans remplacer les pays dans la priorisation finale',
        },
        {
            'page_dashboard': 'Page 4 - Controle qualite',
            'composant': 'Carte de synthese des statuts de donnees',
            'granularite_recommandee': 'global',
            'arbitrage': 'rendre visible le hors priorisation, les valeurs manquantes et les exceptions de gouvernance avant toute lecture metier',
        },
        {
            'page_dashboard': 'Page 4 - Controle qualite',
            'composant': 'Table des pays Non classe',
            'granularite_recommandee': 'country-year',
            'arbitrage': 'isoler les lignes sans score calculable afin de ne pas les confondre avec une quatrieme priorite metier',
        },
        {
            'page_dashboard': 'Page 4 - Controle qualite',
            'composant': 'Suivi des valeurs manquantes par indicateur',
            'granularite_recommandee': 'indicator-year',
            'arbitrage': 'documenter la couverture reelle des indicateurs qui alimentent les scores et expliquer les absences de classement',
        },
        {
            'page_dashboard': 'Page 4 - Controle qualite',
            'composant': 'Alerte gouvernance China vs composantes',
            'granularite_recommandee': 'country-family-year',
            'arbitrage': 'rappeler qu il ne faut jamais sommer China avec Hong Kong, Macao et Taiwan dans un meme agregat population',
        },
    ]
)

dashboard_design_contract

,page_dashboard,composant,granularite_recommandee,arbitrage
0,Page 2 - Domaines metiers,Courbe temporelle Domaine 1,region,utiliser les regions pour montrer une evolutio...
1,Page 2 - Domaines metiers,Barres empilees structure urbaine/rurale,country,conserver les pays prioritaires car la structu...
2,Page 3 - Priorisation,Top pays et recommandations,country,le pays reste l'unite de decision ; la region ...
3,Page 3 - Priorisation,Vue de contexte macro,region,les regions peuvent resumer les tendances moye...
4,Page 4 - Controle qualite,Carte de synthese des statuts de donnees,global,"rendre visible le hors priorisation, les valeu..."
5,Page 4 - Controle qualite,Table des pays Non classe,country-year,isoler les lignes sans score calculable afin d...
6,Page 4 - Controle qualite,Suivi des valeurs manquantes par indicateur,indicator-year,documenter la couverture reelle des indicateur...
7,Page 4 - Controle qualite,Alerte gouvernance China vs composantes,country-family-year,rappeler qu'il ne faut jamais sommer China ave...


### Implications pour le futur dashboard

Les contrôles de la couche marts n'imposent pas de remplacer les pays par les régions dans toutes les pages du dashboard. Ils conduisent plutôt à un partage des rôles entre granularités : la région sert au cadrage macro et à la lecture temporelle, tandis que le pays reste l'unité d'action pour la priorisation.

Une page dédiée de contrôle qualité est recommandée dans Power BI. Son rôle n'est pas de concurrencer les pages métier, mais de sécuriser leur lecture en exposant explicitement les lignes `Non classe`, la couverture réelle des indicateurs et les exceptions de gouvernance comme le cas China et ses composantes.

In [ ]:
quality_control_contract = pd.DataFrame(
    [
        {
            'bloc_controle': 'Statut de priorisation',
            'question_metier': 'Quelles lignes sont hors priorisation ?',
            'source_recommandee': 'fact_dashboard_star_fr',
            'filtre_ou_mesure': "action_prioritaire = 'Non classe'",
            'usage_pbi': 'page de controle qualite uniquement',
        },
        {
            'bloc_controle': 'Couverture des scores',
            'question_metier': 'Pourquoi un pays n\'est-il pas classe ?',
            'source_recommandee': 'dashboard_water_country_year / priority_snapshot_2016',
            'filtre_ou_mesure': 'compter les valeurs manquantes sur safely_managed, mortality, political_stability et les trois scores',
            'usage_pbi': 'cartes KPI et tableau de diagnostic',
        },
        {
            'bloc_controle': 'Population et gouvernance',
            'question_metier': 'Quelles exceptions ne doivent jamais etre agregees ensemble ?',
            'source_recommandee': 'dashboard_water_country_year',
            'filtre_ou_mesure': 'surveiller China, Hong Kong, Macao, Taiwan dans une meme vue',
            'usage_pbi': 'message d\'alerte ou encart methodologique',
        },
        {
            'bloc_controle': 'Eligibilite snapshot 2016',
            'question_metier': 'Quels pays sont exclus du focus final ?',
            'source_recommandee': 'priority_snapshot_2016 / priority_snapshot_focus_2016',
            'filtre_ou_mesure': 'comparer snapshot total et focus population >= 500000',
            'usage_pbi': 'controle de perimetre avant interpretation',
        },
    ]
)

quality_control_contract

,bloc_controle,question_metier,source_recommandee,filtre_ou_mesure,usage_pbi
0,Statut de priorisation,Quelles lignes sont hors priorisation ?,fact_dashboard_star_fr,action_prioritaire = 'Non classe',page de controle qualite uniquement
1,Couverture des scores,Pourquoi un pays n'est-il pas classe ?,dashboard_water_country_year / priority_snapsh...,compter les valeurs manquantes sur safely_mana...,cartes KPI et tableau de diagnostic
2,Population et gouvernance,Quelles exceptions ne doivent jamais etre agre...,dashboard_water_country_year,"surveiller China, Hong Kong, Macao, Taiwan dan...",message d'alerte ou encart methodologique
3,Eligibilite snapshot 2016,Quels pays sont exclus du focus final ?,priority_snapshot_2016 / priority_snapshot_foc...,comparer snapshot total et focus population >=...,controle de perimetre avant interpretation


: 

## Pipeline reproductible sous GitHub

Pour qu'un pipeline soit reproductible, le dépôt doit contenir :

1. les sources attendues et leur convention de nommage ;
2. les notebooks ou scripts de transformation versionnés ;
3. les règles métier explicites ;
4. les exports intermédiaires et marts quand ils sont assumés comme artefacts ;
5. une séparation nette entre raw, processed et notebooks ;
6. des sorties déterministes, sans dépendance à des manipulations manuelles cachées.

In [17]:
pipeline_contract = pd.DataFrame(
    [
        {
            'layer': 'staging',
            'input': 'data/raw/*.csv',
            'output': 'contrôle qualité et inventaire',
            'contract_note': 'atterrissage brut, sans décision métier définitive',
        },
        {
            'layer': 'intermediate',
            'input': 'raw + règles de nettoyage',
            'output': 'socle canonique harmonisé et conservation des cas métier',
            'contract_note': 'source unique des publications EN, FR et PBI',
        },
        {
            'layer': 'marts_analytics_en',
            'input': 'intermediate',
            'output': 'csv_enrichis/en',
            'contract_note': 'publication analytique en anglais, dérivée sans changer la logique métier',
        },
        {
            'layer': 'marts_analytics_fr',
            'input': 'intermediate',
            'output': 'csv_enrichis/fr',
            'contract_note': 'publication analytique en français, jumelle de la version EN',
        },
        {
            'layer': 'marts_pbi',
            'input': 'intermediate',
            'output': 'pbi_star',
            'contract_note': 'contrat technique séparé pour le modèle Power BI',
        },
    ]
)
pipeline_contract

,layer,input,output,contract_note
0,staging,data/raw/*.csv,contrôle qualité et inventaire,"atterrissage brut, sans décision métier défini..."
1,intermediate,raw + règles de nettoyage,socle canonique harmonisé et conservation des ...,"source unique des publications EN, FR et PBI"
2,marts_analytics_en,intermediate,csv_enrichis/en,"publication analytique en anglais, dérivée san..."
3,marts_analytics_fr,intermediate,csv_enrichis/fr,"publication analytique en français, jumelle de..."
4,marts_pbi,intermediate,pbi_star,contrat technique séparé pour le modèle Power BI


<a id="RNCP37837BC04-PIPELINE"></a>

## Conclusion — Reproductibilité du pipeline

<div style="font-size: 12px; color: #666; margin: 8px 0 4px 0;">
<strong>Compétence :</strong> <span style="background-color: #FFF3E0; padding: 2px 6px; border-radius: 3px;">Gérer la documentation</span> · <span style="background-color: #FFF3E0; padding: 2px 6px; border-radius: 3px;">Organiser un projet data</span>
</div>